# Messy Folder Organizer — Exact Copy Version

Goal:
- Read your inventory / excluded CSV or Excel files.
- Categorize each original image into:
  - `usable/` → blank reason
  - `unusable/<reason>/` → `polish`, `b/w`, `occluded`, `pathology`, `blurry`
  - `anomaly/<reason>/` → `anomaly`, `zero_bytes`
- Copy files from the original folder **without opening, resizing, re-saving, compressing, brightening, or changing image bytes**.

This notebook uses `shutil.copy2`, so the image content is copied exactly. It also verifies copied files by file size and SHA-256 hash.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import hashlib
import re
from collections import defaultdict
from datetime import datetime

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

## v2 fix: zero-byte originals

This version also scans `MAIN_FOLDER` for zero-byte image files that are **not listed** in the CSV/Excel files, and adds them to `anomaly/zero_bytes/`. They are copied with `shutil.copy2`, so even empty files are preserved exactly.

## 1) User settings

Change these paths only.

In [11]:
# =========================
# CHANGE THESE PATHS
# =========================

# Main messy/original folder containing the actual images.
MAIN_FOLDER = Path(r'/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5')

# Your CSV or Excel files with image names + reasons.
# You can use .csv, .xlsx, or .xls files here.
ANNOTATION_FILES = [
    Path(r'/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/excluded.csv'),
    Path(r'/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/inventory_saved.csv'),
]

# Output folder. This notebook COPIES into this folder and does not modify MAIN_FOLDER.
OUTPUT_FOLDER = MAIN_FOLDER.parent / f"{MAIN_FOLDER.name}_organized_exact_copy"

# Start with True first. It previews what would happen without copying.
# After checking the preview, set to False and rerun.
DRY_RUN = False

# Keep original folder structure inside each category to avoid filename collision.
# Example: output/unusable/polish/subfolder/image.jpeg
PRESERVE_RELATIVE_PATHS = True

# If False, existing output files are not overwritten.
ALLOW_OVERWRITE = False

# Usually keep this True so every copied file is checked byte-for-byte.
VERIFY_AFTER_COPY = True

# IMPORTANT: also copy zero-byte image files found in MAIN_FOLDER even if they are not listed in the CSV/Excel files.
# These will go to: anomaly/zero_bytes/
SCAN_UNLISTED_ZERO_BYTE_FILES = True

print("MAIN_FOLDER:", MAIN_FOLDER)
print("OUTPUT_FOLDER:", OUTPUT_FOLDER)
print("DRY_RUN:", DRY_RUN)

MAIN_FOLDER: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5
OUTPUT_FOLDER: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy
DRY_RUN: False


## 2) Helper functions

In [12]:
IMAGE_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp",
    ".heic", ".heif"
}

def read_annotation_file(path: Path) -> pd.DataFrame:
    """Read CSV/Excel robustly. Tries common encodings for CSV."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Annotation file not found: {path}")

    suffix = path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    elif suffix == ".csv":
        last_error = None
        for enc in ["utf-8-sig", "utf-8", "big5", "cp950", "latin1"]:
            try:
                df = pd.read_csv(path, encoding=enc)
                print(f"Read {path.name} with encoding={enc}, rows={len(df)}")
                break
            except Exception as e:
                last_error = e
        else:
            raise last_error
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}. Use CSV/XLSX/XLS.")

    df.columns = [str(c).strip() for c in df.columns]
    df["source_file"] = path.name
    return df


def clean_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_stem_from_basename(name: str) -> str:
    name = clean_text(name)
    if not name:
        return ""
    return Path(name).stem.lower()


def split_reasons(reason_text: str):
    """Split multi-reason strings like 'occluded, anomaly'."""
    reason_text = clean_text(reason_text).lower()
    if not reason_text:
        return []
    parts = re.split(r"[,;/|]+", reason_text)
    return [p.strip() for p in parts if p.strip()]


def normalize_one_reason(reason: str) -> str:
    """Map messy labels to the target reason names."""
    r = clean_text(reason).lower()
    r = re.sub(r"\s+", " ", r)

    # Treat unclear/manual unknown as anomaly so it does not silently enter usable.
    # You can edit this if you want unknown files handled differently.
    if r in {"anomaly", "unknown", "weird", "corrupt", "corrupted"}:
        return "anomaly"
    if "zero" in r and "byte" in r:
        return "zero_bytes"
    if r in {"polish", "clear polish", "nail polish"}:
        return "polish"
    if "polish" in r:
        return "polish"
    if r in {"b/w", "bw", "b&w", "black white", "black/white", "black and white", "grayscale", "grey scale"}:
        return "b/w"
    if r in {"occluded", "occlusion", "blocked", "covered"}:
        return "occluded"
    if r in {"pathology", "disease", "abnormal nail"}:
        return "pathology"
    if r in {"blurry", "blur", "out of focus", "unfocused"}:
        return "blurry"
    return r


UNUSABLE_REASONS = {"polish", "b/w", "occluded", "pathology", "blurry"}
ANOMALY_REASONS = {"anomaly", "zero_bytes"}

def classify_reason(reason_text: str, actual_size_bytes=None, csv_size_bytes=None):
    """
    Return:
    - top_folder: usable / unusable / anomaly
    - reason_folder: blank for usable, otherwise normalized reason
    - normalized_reasons: list of normalized reasons found
    - status_note: explanation/warning
    """
    # Actual zero-byte file always wins.
    if actual_size_bytes == 0:
        return "anomaly", "zero_bytes", ["zero_bytes"], "actual file size is 0 bytes"

    # CSV zero-byte fallback when source cannot be checked yet.
    if actual_size_bytes is None:
        try:
            if pd.notna(csv_size_bytes) and int(csv_size_bytes) == 0:
                return "anomaly", "zero_bytes", ["zero_bytes"], "CSV size_bytes is 0"
        except Exception:
            pass

    parts = split_reasons(reason_text)
    normalized = [normalize_one_reason(p) for p in parts]
    normalized = [r for r in normalized if r]

    if not normalized:
        return "usable", "", [], "blank reason"

    # If any anomaly reason is present, keep it in anomaly.
    if any(r in ANOMALY_REASONS for r in normalized):
        chosen = "zero_bytes" if "zero_bytes" in normalized else "anomaly"
        return "anomaly", chosen, normalized, "contains anomaly reason"

    # If any unusable reason is present, keep it in unusable.
    unusable_found = [r for r in normalized if r in UNUSABLE_REASONS]
    if unusable_found:
        # Priority order keeps output folders stable.
        for preferred in ["polish", "b/w", "occluded", "pathology", "blurry"]:
            if preferred in unusable_found:
                return "unusable", preferred, normalized, "contains unusable reason"

    # Anything unmapped should not silently become usable.
    return "anomaly", "anomaly", normalized, f"unmapped reason treated as anomaly: {normalized}"


def sha256_file(path: Path, chunk_size=1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def safe_relpath(path: Path, root: Path) -> Path:
    try:
        return path.resolve().relative_to(root.resolve())
    except Exception:
        return Path(path.name)


def make_unique_path(dest: Path) -> Path:
    """Avoid overwriting by adding __dup001 before extension."""
    if not dest.exists():
        return dest
    stem, suffix = dest.stem, dest.suffix
    parent = dest.parent
    for i in range(1, 10000):
        candidate = parent / f"{stem}__dup{i:03d}{suffix}"
        if not candidate.exists():
            return candidate
    raise RuntimeError(f"Too many duplicate names for {dest}")

## 3) Load and combine the annotation files

In [13]:
raw_dfs = []
for f in ANNOTATION_FILES:
    df = read_annotation_file(f)
    raw_dfs.append(df)

raw = pd.concat(raw_dfs, ignore_index=True, sort=False)

# Drop completely empty unnamed columns often created by spreadsheet exports.
unnamed_cols = [c for c in raw.columns if c.lower().startswith("unnamed")]
for c in unnamed_cols:
    if raw[c].isna().all():
        raw = raw.drop(columns=[c])

# Create standard columns even if different files use different names.
def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

basename_col = first_existing_col(raw, ["basename", "filename", "file_name", "image_name", "name"])
stem_col     = first_existing_col(raw, ["stem", "stem_lower"])
relpath_col  = first_existing_col(raw, ["relpath", "relpath_from_original", "relative_path"])
abspath_col  = first_existing_col(raw, ["abspath", "abs_path", "absolute_path"])
reason_col   = first_existing_col(raw, ["reason", "reasons", "exclude_reason"])
notes_col    = first_existing_col(raw, ["notes", "note", "comment", "comments"])
size_col     = first_existing_col(raw, ["size_bytes", "bytes", "file_size"])

if basename_col is None and stem_col is None:
    raise ValueError("Could not find an image-name column. Need one of: basename, filename, image_name, stem.")

raw["basename_std"] = raw[basename_col].map(clean_text) if basename_col else ""
raw["stem_std"] = raw[stem_col].map(lambda x: clean_text(x).lower()) if stem_col else ""
raw.loc[raw["stem_std"].eq("") & raw["basename_std"].ne(""), "stem_std"] = raw.loc[
    raw["stem_std"].eq("") & raw["basename_std"].ne(""), "basename_std"
].map(normalize_stem_from_basename)

raw["reason_std"] = raw[reason_col].map(clean_text) if reason_col else ""
raw["notes_std"] = raw[notes_col].map(clean_text) if notes_col else ""
raw["relpath_std"] = raw[relpath_col].map(clean_text) if relpath_col else ""
raw["abspath_std"] = raw[abspath_col].map(clean_text) if abspath_col else ""
raw["csv_size_bytes"] = raw[size_col] if size_col else np.nan

# Prefer rows with a nonblank reason when multiple sheets mention the same image.
# This lets excluded.csv override blank inventory rows.
raw["has_reason"] = raw["reason_std"].ne("")
raw["priority"] = raw["has_reason"].astype(int)

def combine_group(g):
    g = g.sort_values(["priority"], ascending=False)
    row = g.iloc[0].copy()

    # Keep first nonblank basename/relpath/abspath if selected row lacks it.
    for col in ["basename_std", "relpath_std", "abspath_std", "notes_std"]:
        if clean_text(row.get(col, "")) == "":
            vals = [clean_text(v) for v in g[col].tolist() if clean_text(v)]
            if vals:
                row[col] = vals[0]

    # Combine unique nonblank reasons if multiple appear.
    reasons = []
    for r in g["reason_std"].tolist():
        r = clean_text(r)
        if r and r not in reasons:
            reasons.append(r)
    row["reason_std"] = ", ".join(reasons)
    row["all_source_files"] = ", ".join(sorted(set(g["source_file"].astype(str))))
    row["duplicate_rows_merged"] = len(g)
    return row

records = (
    raw[raw["stem_std"].ne("")]
    .groupby("stem_std", group_keys=False)
    .apply(combine_group)
    .reset_index(drop=True)
)

print("Raw rows:", len(raw))
print("Unique image records:", len(records))
display(records[["stem_std", "basename_std", "reason_std", "notes_std", "all_source_files", "duplicate_rows_merged"]].head(20))

Read excluded.csv with encoding=big5, rows=104
Read inventory_saved.csv with encoding=utf-8-sig, rows=862
Raw rows: 966
Unique image records: 966


/var/folders/gc/qn5t97b94vv1hblf5v115kp80000gn/T/ipykernel_88422/1770830380.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_group)


,stem_std,basename_std,reason_std,notes_std,all_source_files,duplicate_rows_merged
0,20210607132915_pid2625_eowkbq,20210607132915_pid2625_eowkBq.jpg,,,inventory_saved.csv,1
1,20210607135137_pid2625_5ae83w,20210607135137_pid2625_5Ae83W.jpg,,,inventory_saved.csv,1
2,20210607141619_pid2625_fuca8f,20210607141619_pid2625_FuCA8f.jpg,,,inventory_saved.csv,1
3,20210607143057_pid2625_4x464c,20210607143057_pid2625_4X464c.jpg,,,inventory_saved.csv,1
4,20210607160023_pid2625_smjxnc,20210607160023_pid2625_SmjXnC.jpg,,,inventory_saved.csv,1
5,20210607162341_pid2625_inzaot,20210607162341_pid2625_InZAoT.jpg,,,inventory_saved.csv,1
6,20210607163405_pid2625_fmmkvl,20210607163405_pid2625_FMMKVL.jpg,,,inventory_saved.csv,1
7,20210607170118_pid2625_jxy7qq,20210607170118_pid2625_jxY7qQ.jpg,,,inventory_saved.csv,1
8,20210608131556_pid2625_iyury3,20210608131556_pid2625_iYUrY3.jpg,,,inventory_saved.csv,1
9,20210608144618_pid2625_wiwsps,20210608144618_pid2625_WIWsPS.jpg,,,inventory_saved.csv,1


## 4) Index the original folder and resolve each CSV row to a real file

In [14]:
if not MAIN_FOLDER.exists():
    raise FileNotFoundError(f"MAIN_FOLDER does not exist: {MAIN_FOLDER}")

# Index files recursively under MAIN_FOLDER.
all_files = [p for p in MAIN_FOLDER.rglob("*") if p.is_file()]
image_files = [p for p in all_files if p.suffix.lower() in IMAGE_EXTS]

print("All files found:", len(all_files))
print("Image-like files found:", len(image_files))

by_basename = defaultdict(list)
by_stem = defaultdict(list)
by_relpath = {}

for p in all_files:
    rel = safe_relpath(p, MAIN_FOLDER)
    by_relpath[str(rel).replace("\\", "/").lower()] = p
    by_basename[p.name.lower()].append(p)
    by_stem[p.stem.lower()].append(p)

def resolve_source_path(row):
    """Find the actual original file for one annotation row."""
    # 1) Try relpath from CSV under MAIN_FOLDER.
    rel = clean_text(row.get("relpath_std", ""))
    if rel:
        rel_norm = rel.replace("\\", "/").lower()
        candidate = MAIN_FOLDER / rel
        if candidate.exists() and candidate.is_file():
            return candidate, "relpath"
        if rel_norm in by_relpath:
            return by_relpath[rel_norm], "relpath_index"

    # 2) Try basename exact match.
    base = clean_text(row.get("basename_std", ""))
    if base:
        matches = by_basename.get(base.lower(), [])
        if len(matches) == 1:
            return matches[0], "basename"
        if len(matches) > 1:
            return matches[0], f"basename_multiple_matches_{len(matches)}"

    # 3) Try stem match.
    stem = clean_text(row.get("stem_std", "")).lower()
    if stem:
        matches = by_stem.get(stem, [])
        if len(matches) == 1:
            return matches[0], "stem"
        if len(matches) > 1:
            return matches[0], f"stem_multiple_matches_{len(matches)}"

    # 4) Try absolute path only if it exists on this machine.
    abs_p = clean_text(row.get("abspath_std", ""))
    if abs_p:
        p = Path(abs_p)
        if p.exists() and p.is_file():
            return p, "abspath"

    return None, "not_found"

resolved_paths = []
resolve_methods = []

for _, row in records.iterrows():
    p, method = resolve_source_path(row)
    resolved_paths.append(p)
    resolve_methods.append(method)

records["source_path"] = [str(p) if p else "" for p in resolved_paths]
records["resolve_method"] = resolve_methods
records["actual_size_bytes"] = [Path(p).stat().st_size if p else np.nan for p in resolved_paths]

print(records["resolve_method"].value_counts(dropna=False))
missing = records[records["source_path"].eq("")]
print("Missing files:", len(missing))

display(records[["basename_std", "reason_std", "source_path", "resolve_method", "actual_size_bytes"]].head(20))

if len(missing):
    display(missing[["stem_std", "basename_std", "reason_std", "relpath_std", "abspath_std", "resolve_method"]].head(50))

All files found: 1008
Image-like files found: 1007
resolve_method
relpath     862
basename    104
Name: count, dtype: int64
Missing files: 0


,basename_std,reason_std,source_path,resolve_method,actual_size_bytes
0,20210607132915_pid2625_eowkBq.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607132915_pid2625_eowkBq.jpg,relpath,1507505
1,20210607135137_pid2625_5Ae83W.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607135137_pid2625_5Ae83W.jpg,relpath,1831450
2,20210607141619_pid2625_FuCA8f.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607141619_pid2625_FuCA8f.jpg,relpath,1052814
3,20210607143057_pid2625_4X464c.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607143057_pid2625_4X464c.jpg,relpath,1390694
4,20210607160023_pid2625_SmjXnC.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607160023_pid2625_SmjXnC.jpg,relpath,1474882
5,20210607162341_pid2625_InZAoT.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607162341_pid2625_InZAoT.jpg,relpath,1323913
6,20210607163405_pid2625_FMMKVL.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607163405_pid2625_FMMKVL.jpg,relpath,1616383
7,20210607170118_pid2625_jxY7qQ.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607170118_pid2625_jxY7qQ.jpg,relpath,1298092
8,20210608131556_pid2625_iYUrY3.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210608131556_pid2625_iYUrY3.jpg,relpath,1369805
9,20210608144618_pid2625_WIWsPS.jpg,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210608144618_pid2625_WIWsPS.jpg,relpath,1225563


In [15]:
# =========================
# Add zero-byte original images that were not listed in CSV/Excel
# =========================
# Why this exists:
# Some zero-byte files may not appear in excluded.csv or inventory_saved.csv.
# We still want them copied into anomaly/zero_bytes because they are part of the original messy folder.

if SCAN_UNLISTED_ZERO_BYTE_FILES:
    resolved_source_paths = set(records["source_path"].map(clean_text))

    zero_byte_files = []
    for p in image_files:
        try:
            if p.stat().st_size == 0:
                zero_byte_files.append(p)
        except FileNotFoundError:
            pass

    unlisted_zero_byte_files = [
        p for p in zero_byte_files
        if str(p) not in resolved_source_paths
    ]

    print("Zero-byte image files found in MAIN_FOLDER:", len(zero_byte_files))
    print("Zero-byte image files not already in CSV/Excel records:", len(unlisted_zero_byte_files))

    if unlisted_zero_byte_files:
        new_rows = []
        for p in unlisted_zero_byte_files:
            rel = safe_relpath(p, MAIN_FOLDER)
            row = {col: "" for col in records.columns}
            row.update({
                "stem_std": p.stem.lower(),
                "basename_std": p.name,
                "reason_std": "zero_bytes",
                "notes_std": "auto-detected zero-byte file from original folder",
                "relpath_std": str(rel).replace("\\", "/"),
                "abspath_std": str(p),
                "csv_size_bytes": np.nan,
                "has_reason": True,
                "priority": 1,
                "source_file": "AUTO_SCAN_ZERO_BYTES",
                "all_source_files": "AUTO_SCAN_ZERO_BYTES",
                "duplicate_rows_merged": 1,
                "source_path": str(p),
                "resolve_method": "auto_zero_byte_scan",
                "actual_size_bytes": 0,
            })
            new_rows.append(row)

        records = pd.concat([records, pd.DataFrame(new_rows)], ignore_index=True, sort=False)

        display(pd.DataFrame({
            "zero_byte_file_added": [p.name for p in unlisted_zero_byte_files[:50]],
            "source_path": [str(p) for p in unlisted_zero_byte_files[:50]],
        }))
else:
    print("SCAN_UNLISTED_ZERO_BYTE_FILES=False, so only CSV/Excel-listed files are used.")


Zero-byte image files found in MAIN_FOLDER: 43
Zero-byte image files not already in CSV/Excel records: 41


,zero_byte_file_added,source_path
0,20210701131914_pid2625_9NJTei.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210701131914_pid2625_9NJTei.jpg
1,20210630164248_pid2625_GtYap9.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210630164248_pid2625_GtYap9.jpg
2,20210713163949_pid2625_bCraVA.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210713163949_pid2625_bCraVA.jpg
3,20210629145732_pid2625_985urE.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210629145732_pid2625_985urE.jpg
4,20210709165700_pid2625_uCjVbj.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210709165700_pid2625_uCjVbj.jpg
5,20210701135425_pid2625_gIogZd.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210701135425_pid2625_gIogZd.jpg
6,20210701124802_pid2625_9niBUk.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210701124802_pid2625_9niBUk.jpg
7,20210712163551_pid2625_JRDZhK.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210712163551_pid2625_JRDZhK.jpg
8,20210629122158_pid2625_sIaNIp.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210629122158_pid2625_sIaNIp.jpg
9,20210629164800_pid2625_nKx4RE.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210629164800_pid2625_nKx4RE.jpg


## 5) Classify into `usable`, `unusable`, and `anomaly`

In [16]:
class_rows = []
for _, row in records.iterrows():
    actual_size = row["actual_size_bytes"]
    actual_size = None if pd.isna(actual_size) else int(actual_size)

    top, reason_folder, normalized_reasons, note = classify_reason(
        row["reason_std"],
        actual_size_bytes=actual_size,
        csv_size_bytes=row.get("csv_size_bytes", np.nan),
    )
    class_rows.append((top, reason_folder, "|".join(normalized_reasons), note))

records[["top_folder", "reason_folder", "normalized_reasons", "classification_note"]] = pd.DataFrame(
    class_rows, index=records.index
)

print("Top folder counts:")
display(records["top_folder"].value_counts(dropna=False).rename_axis("folder").reset_index(name="count"))

print("Reason folder counts:")
display(records.groupby(["top_folder", "reason_folder"], dropna=False).size().reset_index(name="count"))

print("Unmapped / warning notes:")
display(records[records["classification_note"].str.contains("unmapped", na=False)][
    ["basename_std", "reason_std", "normalized_reasons", "classification_note"]
].head(50))

Top folder counts:


,folder,count
0,usable,836
1,unusable,121
2,anomaly,50


Reason folder counts:


,top_folder,reason_folder,count
0,anomaly,anomaly,7
1,anomaly,zero_bytes,43
2,unusable,blurry,3
3,unusable,occluded,3
4,unusable,pathology,7
5,unusable,polish,108
6,usable,,836


Unmapped / warning notes:


,basename_std,reason_std,normalized_reasons,classification_note
787,20221115103130_pid2625_zTEoFR.jpeg,b/w,b|w,"unmapped reason treated as anomaly: ['b', 'w']"
792,20221115122636_pid2625_zVKthD.jpeg,b/w,b|w,"unmapped reason treated as anomaly: ['b', 'w']"
796,20221116100613_pid2625_7Ckw9S.jpeg,b/w,b|w,"unmapped reason treated as anomaly: ['b', 'w']"


## 6) Preview destination paths

In [17]:
def build_destination_path(row):
    src = Path(row["source_path"])

    top = row["top_folder"]
    reason = clean_text(row["reason_folder"])

    if PRESERVE_RELATIVE_PATHS:
        inner = safe_relpath(src, MAIN_FOLDER)
    else:
        inner = Path(src.name)

    if top == "usable":
        dest = OUTPUT_FOLDER / "usable" / inner
    else:
        dest = OUTPUT_FOLDER / top / reason / inner

    return dest

records["dest_path"] = ""
found_mask = records["source_path"].ne("")
records.loc[found_mask, "dest_path"] = records.loc[found_mask].apply(
    lambda r: str(build_destination_path(r)), axis=1
)

display(records[[
    "basename_std", "reason_std", "top_folder", "reason_folder",
    "source_path", "dest_path", "classification_note"
]].head(30))

print("Ready to copy:", found_mask.sum())
print("Not found:", (~found_mask).sum())

,basename_std,reason_std,top_folder,reason_folder,source_path,dest_path,classification_note
0,20210607132915_pid2625_eowkBq.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607132915_pid2625_eowkBq.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
1,20210607135137_pid2625_5Ae83W.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607135137_pid2625_5Ae83W.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
2,20210607141619_pid2625_FuCA8f.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607141619_pid2625_FuCA8f.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
3,20210607143057_pid2625_4X464c.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607143057_pid2625_4X464c.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
4,20210607160023_pid2625_SmjXnC.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607160023_pid2625_SmjXnC.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
5,20210607162341_pid2625_InZAoT.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607162341_pid2625_InZAoT.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
6,20210607163405_pid2625_FMMKVL.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607163405_pid2625_FMMKVL.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
7,20210607170118_pid2625_jxY7qQ.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210607170118_pid2625_jxY7qQ.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
8,20210608131556_pid2625_iYUrY3.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210608131556_pid2625_iYUrY3.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason
9,20210608144618_pid2625_WIWsPS.jpg,,usable,,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5/20210608144618_pid2625_WIWsPS.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable/2021060...,blank reason


Ready to copy: 1007
Not found: 0


## 7) Copy exactly and verify byte-identical output

When the preview looks right, set `DRY_RUN = False` in the settings cell and rerun.

In [18]:
copy_report = []

for _, row in records.iterrows():
    src_text = clean_text(row["source_path"])
    if not src_text:
        copy_report.append({
            "status": "missing_source",
            "source_path": "",
            "dest_path": "",
            "top_folder": row["top_folder"],
            "reason_folder": row["reason_folder"],
            "basename": row["basename_std"],
            "reason_original": row["reason_std"],
            "same_size": False,
            "same_sha256": False,
            "src_size": np.nan,
            "dest_size": np.nan,
            "src_sha256": "",
            "dest_sha256": "",
        })
        continue

    src = Path(src_text)
    dest = build_destination_path(row)

    if dest.exists() and not ALLOW_OVERWRITE:
        dest = make_unique_path(dest)

    status = "dry_run" if DRY_RUN else "copied"

    src_size = src.stat().st_size
    dest_size = np.nan
    src_hash = ""
    dest_hash = ""
    same_size = False
    same_hash = False

    if not DRY_RUN:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)  # exact byte copy; no image decoding or re-saving

        dest_size = dest.stat().st_size
        same_size = (src_size == dest_size)

        if VERIFY_AFTER_COPY:
            src_hash = sha256_file(src)
            dest_hash = sha256_file(dest)
            same_hash = (src_hash == dest_hash)
        else:
            same_hash = np.nan
    else:
        same_size = True
        same_hash = np.nan

    copy_report.append({
        "status": status,
        "source_path": str(src),
        "dest_path": str(dest),
        "top_folder": row["top_folder"],
        "reason_folder": row["reason_folder"],
        "basename": src.name,
        "reason_original": row["reason_std"],
        "normalized_reasons": row["normalized_reasons"],
        "classification_note": row["classification_note"],
        "resolve_method": row["resolve_method"],
        "same_size": same_size,
        "same_sha256": same_hash,
        "src_size": src_size,
        "dest_size": dest_size,
        "src_sha256": src_hash,
        "dest_sha256": dest_hash,
    })

copy_report_df = pd.DataFrame(copy_report)

print("Copy status:")
display(copy_report_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))

print("Category counts:")
display(copy_report_df.groupby(["top_folder", "reason_folder"], dropna=False).size().reset_index(name="count"))

if not DRY_RUN:
    problems = copy_report_df[
        (copy_report_df["status"].ne("copied")) |
        (copy_report_df["same_size"] != True) |
        (copy_report_df["same_sha256"] != True)
    ]
    print("Verification problems:", len(problems))
    display(problems.head(50))
else:
    print("DRY_RUN=True, so no files were copied yet.")

Copy status:


,status,count
0,copied,1007


Category counts:


,top_folder,reason_folder,count
0,anomaly,anomaly,7
1,anomaly,zero_bytes,43
2,unusable,blurry,3
3,unusable,occluded,3
4,unusable,pathology,7
5,unusable,polish,108
6,usable,,836


Verification problems: 0


,status,source_path,dest_path,top_folder,reason_folder,basename,reason_original,normalized_reasons,classification_note,resolve_method,same_size,same_sha256,src_size,dest_size,src_sha256,dest_sha256


## 8) Save reports

In [19]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
REPORT_FOLDER = OUTPUT_FOLDER / "_reports"
REPORT_FOLDER.mkdir(parents=True, exist_ok=True)

records_report_path = REPORT_FOLDER / f"resolved_classification_report_{timestamp}.csv"
copy_report_path = REPORT_FOLDER / f"copy_verification_report_{timestamp}.csv"

records.to_csv(records_report_path, index=False, encoding="utf-8-sig")
copy_report_df.to_csv(copy_report_path, index=False, encoding="utf-8-sig")

print("Saved:")
print(records_report_path)
print(copy_report_path)

# Useful final checks
print("\nExpected final top-level folders:")
for folder in ["usable", "unusable", "anomaly"]:
    print("-", OUTPUT_FOLDER / folder)

if not DRY_RUN:
    print("\nFinal copied file counts by folder:")
    for folder in ["usable", "unusable", "anomaly"]:
        folder_path = OUTPUT_FOLDER / folder
        n = sum(1 for p in folder_path.rglob("*") if p.is_file()) if folder_path.exists() else 0
        print(folder, n)

Saved:
/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/_reports/resolved_classification_report_20260619_223823.csv
/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/_reports/copy_verification_report_20260619_223823.csv

Expected final top-level folders:
- /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/usable
- /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/unusable
- /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/anomaly

Final copied file counts by folder:
usable 836
unusable 121
anomaly 50
